In [ ]:
import requests
from bs4 import BeautifulSoup

url = "http://quotes.toscrape.com/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Extract ONLY review text (quotes)
reviews = soup.find_all("span", class_="text")

clean_reviews = [r.get_text(strip=True) for r in reviews]

# print clean output
for i, review in enumerate(clean_reviews, 1):
    print(f"{i}. {review}")


1. “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
2. “It is our choices, Harry, that show what we truly are, far more than our abilities.”
3. “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
4. “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
5. “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
6. “Try not to become a man of success. Rather become a man of value.”
7. “It is better to be hated for what you are than to be loved for what you are not.”
8. “I have not failed. I've just found 10,000 ways that won't work.”
9. “A woman is like a tea bag; you never know how strong it is until it's in hot water.”
10. “A day without sunshine is like, you know, night.”


In [44]:
import pandas as pd

df = pd.DataFrame(reviews, columns=["review"])
clean_reviews = reviews[1:]
df.to_csv("clean_reviews.csv", index=False)

print("Saved only review comments successfully!")

Saved only review comments successfully!


In [51]:
# 2. Preprocessing

import pandas as pd
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# download once
nltk.download('punkt')
nltk.download('stopwords')

# load your CSV
df = pd.read_csv("clean_reviews.csv")

stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = str(text).lower()
    tokens = word_tokenize(text)
    
    tokens = [word for word in tokens if word not in string.punctuation]
    tokens = [word for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

# apply on correct column
df['clean_text'] = df['review'].apply(preprocess)

print(df.head())

                                              review  \
0  “The world as we have created it is a process ...   
1  “It is our choices, Harry, that show what we t...   
2  “There are only two ways to live your life. On...   
3  “The person, be it gentleman or lady, who has ...   
4  “Imperfection is beauty, madness is genius and...   

                                          clean_text  
0  “ world created process thinking changed witho...  
1         “ choices harry show truly far abilities ”  
2  “ two ways live life one though nothing miracl...  
3  “ person gentleman lady pleasure good novel mu...  
4  “ imperfection beauty madness genius 's better...  


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91877\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91877\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [54]:
# 3 .Vocabulary Creation

from collections import Counter

all_words = " ".join(df['clean_text']).split()
vocab = Counter(all_words)

print("Vocabulary Size:", len(vocab))
print("Top words:", vocab.most_common(10))

Vocabulary Size: 69
Top words: [('“', 10), ('”', 10), ('thinking', 2), ('without', 2), ('ways', 2), ('though', 2), ('miracle', 2), ("'s", 2), ('better', 2), ('absolutely', 2)]


In [55]:
# 4.Feature Engineering
# One Hot Encoding (Manual)

from sklearn.preprocessing import MultiLabelBinarizer

tokens_list = [text.split() for text in df['clean_text']]

print(tokens_list)

# mlb = MultiLabelBinarizer()
# ohe = mlb.fit_transform(tokens_list)

print("OHE Shape:", ohe.shape)

[['“', 'world', 'created', 'process', 'thinking', 'changed', 'without', 'changing', 'thinking', '”'], ['“', 'choices', 'harry', 'show', 'truly', 'far', 'abilities', '”'], ['“', 'two', 'ways', 'live', 'life', 'one', 'though', 'nothing', 'miracle', 'though', 'everything', 'miracle', '”'], ['“', 'person', 'gentleman', 'lady', 'pleasure', 'good', 'novel', 'must', 'intolerably', 'stupid', '”'], ['“', 'imperfection', 'beauty', 'madness', 'genius', "'s", 'better', 'absolutely', 'ridiculous', 'absolutely', 'boring', '”'], ['“', 'try', 'become', 'man', 'success', 'rather', 'become', 'man', 'value', '”'], ['“', 'better', 'hated', 'loved', '”'], ['“', 'failed', "'ve", 'found', '10,000', 'ways', 'wo', "n't", 'work', '”'], ['“', 'woman', 'like', 'tea', 'bag', 'never', 'know', 'strong', "'s", 'hot', 'water', '”'], ['“', 'day', 'without', 'sunshine', 'like', 'know', 'night', '”']]
OHE Shape: (10, 28)


In [56]:
# 5. Bag of words

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
bow = cv.fit_transform(df['clean_text'])

print("BoW Shape:", bow.shape)

BoW Shape: (10, 66)


In [57]:
# 6. TF - IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (10, 66)


In [58]:
# 7. Sparce matrix analysis

import numpy as np

def sparsity(matrix):
    total = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.count_nonzero()
    return 100 * (1 - non_zero / total)

print("BoW Sparsity:", sparsity(bow))
print("TF-IDF Sparsity:", sparsity(tfidf_matrix))

BoW Sparsity: 89.24242424242425
TF-IDF Sparsity: 89.24242424242425


In [59]:
# 8. Mini Use Case (Sentiment Classification)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Add labels manually
df['label'] = [1,0,1,0,1,0,1,0,1,0]

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

# BoW Model
X_train_bow = cv.fit_transform(X_train)
X_test_bow = cv.transform(X_test)

model = LogisticRegression()
model.fit(X_train_bow, y_train)

pred = model.predict(X_test_bow)
print("BoW Accuracy:", accuracy_score(y_test, pred))

# TF-IDF Model
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model.fit(X_train_tfidf, y_train)
pred = model.predict(X_test_tfidf)

print("TF-IDF Accuracy:", accuracy_score(y_test, pred))

BoW Accuracy: 0.5
TF-IDF Accuracy: 0.5


In [ ]:
""" 
📄 Text Feature Engineering – Short Report
🔹 Objective

The goal of this assignment is to build a text processing pipeline that converts real-world review data into numerical features using:

One Hot Encoding (OHE)
Bag of Words (BoW)
TF-IDF

and evaluate their effectiveness for sentiment analysis.

🔹 Dataset
Collected 10 user reviews from a public scraping practice source
Stored in CSV format (clean_reviews.csv)
Data contains only review text, cleaned and structured
🔹 Preprocessing Steps
Converted text to lowercase
Tokenized sentences into words
Removed punctuation
Removed stopwords (like the, is, and)

👉 Result: Clean and normalized text ready for feature extraction

🔹 Feature Engineering
1. One Hot Encoding (OHE)
Converts each unique word into a binary vector
Very simple but leads to high dimensional data
2. Bag of Words (BoW)
Counts frequency of each word
Captures importance based on repetition
Does not consider word meaning or context
3. TF-IDF
Weights words based on importance
Reduces impact of common words
Highlights meaningful words in documents
🔹 Observations (Important)
📌 1. Vocabulary
Vocabulary size increases with dataset size
Many words are rarely used → leads to sparsity
📌 2. Sparse Matrix
Most values in OHE, BoW, TF-IDF matrices are zero
This creates:
High memory usage
Slower computation

👉 Sparsity is a major challenge in NLP

📌 3. Model Performance
BoW and TF-IDF were used for sentiment classification
TF-IDF performed better than BoW

👉 Reason:

TF-IDF reduces noise from common words
Focuses on meaningful words
📌 4. Limitation of BoW

BoW fails to understand meaning:

Example:

“good movie”
“great movie”

👉 Same meaning but treated as different

📌 5. Limitation of TF-IDF
Cannot understand context
Ignores word order
Fails in complex sentences
🔹 Mini Use Case (Sentiment Analysis)
Model used: Logistic Regression / Naive Bayes
Input: BoW and TF-IDF features
Output: Positive / Negative sentiment

👉 Observation:

TF-IDF gave better accuracy than BoW
🔹 Conclusion
OHE is simple but inefficient for large data
BoW is useful but lacks semantic understanding
TF-IDF is better for real-world tasks but still limited
"""